In [1]:
import os
import xarray as xr
import dask
import dask.distributed as Client #cdj
# from dask.diagnostics import ProgressBar
from progress import ProgressBar
import hydra
from omegaconf import DictConfig
import logging
import numpy as np


In [2]:
import hydra
from omegaconf import DictConfig
# 初始化 Hydra
hydra.initialize(config_path="conf", version_base=None)
# 组合配置
cfg = hydra.compose(config_name="config_uvsp_swh_mwp")
cfg

{'zarr_store_path': '/datasets/zarr_data_uvsp_swh_mwp', 'hdf5_store_path': '/datasets/hdf5_data_uvsp_swh_mwp_1', 'dt': 1, 'start_train_year': 1987, 'end_train_year': 2017, 'test_years': [2018, 2019], 'out_of_sample_years': [2020], 'compute_mean_std': True, 'variables': ['10m_u_component_of_wind', '10m_v_component_of_wind', 'surface_pressure', 'significant_height_of_combined_wind_waves_and_swell', 'mean_wave_period']}

In [3]:
	reformated_variables = []
	for variable in cfg.variables:
		if isinstance(variable, str):
			reformated_variables.append(tuple([variable, None]))
		else:
			reformated_variables.append(variable)
   
   	# Return the Zarr paths
	zarr_paths = []
	for variable, pressure_level in reformated_variables:
		zarr_path = f"{cfg.zarr_store_path}/{variable}.zarr"
		zarr_paths.append(zarr_path)
		
	# Check that Zarr arrays have correct dt for time dimension
	for zarr_path in zarr_paths:
		ds = xr.open_zarr(zarr_path)
		time_stamps = ds.time.values
		dt = time_stamps[1:] - time_stamps[:-1]
		assert np.all(
			dt == dt[0]
		), f"Zarr array {zarr_path} has incorrect dt for time dimension. An error may have occurred during download. Please delete the Zarr array and try again."


	zarr_arrays = [xr.open_zarr(path) for path in zarr_paths]
 
	#cdj interpolate the data to the same grid
	reference_lat = zarr_arrays[0].latitude
	reference_lon = zarr_arrays[0].longitude
 

/usr/local/lib/python3.10/dist-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.31.0 or higher is recommended. You are running version 2.24.2
  warnings.warn(


In [4]:
	zarr_arrays = [z.chunk('auto') for z in zarr_arrays]


In [5]:
zarr_arrays[1]

<xarray.Dataset> Size: 66GB
Dimensions:    (latitude: 165, longitude: 169, time: 298056)
Coordinates:
  * latitude   (latitude) float32 660B 42.0 41.75 41.5 41.25 ... 1.5 1.25 1.0
  * longitude  (longitude) float32 676B 98.0 98.25 98.5 ... 139.5 139.8 140.0
  * time       (time) datetime64[ns] 2MB 1987-01-01 ... 2020-12-31T23:00:00
Data variables:
    v10        (time, latitude, longitude) float64 66GB dask.array<chunksize=(601, 165, 169), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6
    history:      2024-08-02 05:39:01 GMT by grib_to_netcdf-2.28.1: /opt/ecmw...

In [6]:
zarr_arrays[3]['swh']

<xarray.DataArray 'swh' (time: 298056, latitude: 83, longitude: 85)> Size: 17GB
dask.array<rechunk-merge, shape=(298056, 83, 85), dtype=float64, chunksize=(2378, 83, 85), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float32 332B 42.0 41.5 41.0 40.5 ... 2.5 2.0 1.5 1.0
  * longitude  (longitude) float32 340B 98.0 98.5 99.0 ... 139.0 139.5 140.0
  * time       (time) datetime64[ns] 2MB 1987-01-01 ... 2020-12-31T23:00:00
Attributes:
    long_name:  Significant height of combined wind waves and swell
    units:      m

In [8]:
zarr_arrays[1]['v10'][:10000]

<xarray.DataArray 'v10' (time: 10000, latitude: 165, longitude: 169)> Size: 2GB
dask.array<getitem, shape=(10000, 165, 169), dtype=float64, chunksize=(601, 165, 169), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float32 660B 42.0 41.75 41.5 41.25 ... 1.5 1.25 1.0
  * longitude  (longitude) float32 676B 98.0 98.25 98.5 ... 139.5 139.8 140.0
  * time       (time) datetime64[ns] 80kB 1987-01-01 ... 1988-02-21T15:00:00
Attributes:
    long_name:  10 metre V wind component
    units:      m s**-1

In [11]:
with dask.config.set(
	scheduler="threads",
	num_workers=64,
	threads_per_worker=2,
	**{"array.slicing.split_large_chunks": False},
):
	mean = zarr_arrays[1]['v10'][:100000].mean(dim=("time", "latitude", "longitude")).compute()
 
mean

<xarray.DataArray 'v10' ()> Size: 8B
array(-0.43434944)

In [12]:
with dask.config.set(
	scheduler="processes",
	num_workers=64,
	threads_per_worker=2,
	**{"array.slicing.split_large_chunks": False},
):
	mean = zarr_arrays[1]['v10'][:100000].mean(dim=("time", "latitude", "longitude")).compute()
 
mean

<xarray.DataArray 'v10' ()> Size: 8B
array(-0.43434944)

In [11]:
		era5_xarray = xr.concat(
			[z[list(z.data_vars.keys())[0]] for z in zarr_arrays], dim="channel"
		)
		era5_xarray = era5_xarray.transpose("time", "channel", "latitude", "longitude")
		era5_xarray.name = "fields"
		era5_xarray = era5_xarray.astype("float32")

In [13]:
		era5_xarray = era5_xarray.chunk('auto')
	
era5_xarray

<xarray.DataArray 'fields' (time: 298056, channel: 5, latitude: 165,
                            longitude: 169)> Size: 166GB
dask.array<rechunk-merge, shape=(298056, 5, 165, 169), dtype=float32, chunksize=(601, 1, 165, 169), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float32 660B 1.0 1.25 1.5 1.75 ... 41.5 41.75 42.0
  * longitude  (longitude) float32 676B 98.0 98.25 98.5 ... 139.5 139.8 140.0
  * time       (time) datetime64[ns] 2MB 1987-01-01 ... 2020-12-31T23:00:00
Dimensions without coordinates: channel
Attributes:
    long_name:  10 metre U wind component
    units:      m s**-1

In [16]:
	with dask.config.set(
		scheduler="processes",
		num_workers=64,
		threads_per_worker=2,
		**{"array.slicing.split_large_chunks": False},
	):
	    mean = era5_xarray.mean(dim=("time", "latitude", "longitude")).values  
    mean

KeyboardInterrupt: 

In [10]:
	with dask.config.set(
		scheduler="processes",
		num_workers=64,
		threads_per_worker=2,
		**{"array.slicing.split_large_chunks": False},
	):
		interp = zarr_arrays[3].interp(latitude=reference_lat, longitude=reference_lon)
		interp.compute()
		# dask.compute(*zarr_arrays)  # 使用 da.compute 计算所有 Dask 数组

KeyboardInterrupt: 